In [0]:
import requests
import json
import os
import time

API_KEY = "0a120f902fa2a44d681830da22eb8ebe"
HEADERS = {"accept":"application/json"}
BASE_URL = "https://api.themoviedb.org/3"

## Extraction & Load to Raw DBFS

In [0]:
# GETTING THE MOVIE IDs
print("Getting movie IDs...")
movie_ids = []

for page in range(1,6):
    discover_url = f"{BASE_URL}/discover/movie?api_key={API_KEY}&with_genres=18,36&language=en-US&page={page}"
    response = requests.get(discover_url, headers=HEADERS)

    if response.status_code == 200:
        data = response.json()
        for movie in data.get("results", []):
            movie_ids.append(movie['id'])

print("Successfully grabbed {len(movie_ids)} movie IDs.")

# PULL DEEP JSON DATA
print("Starting Step 2: Extracting deep JSON for each movie...")
enriched_movies = []

for count, movie_id in enumerate(movie_ids, 1):
    detail_url = f"{BASE_URL}/movie/{movie_id}?api_key={API_KEY}"
    response = requests.get(detail_url, headers=HEADERS)
    
    if response.status_code == 200:
        enriched_movies.append(response.json())
    
    # Ccnsole logging for long jobs
    if count % 20 == 0:
        print(f"Processed {count}/{len(movie_ids)} movies...")
        
    # rate limiter to protect the API
    time.sleep(0.05)

# SAVE TO DATABRICKS STORAGE
# save to databricks file systems (DBFS)
os.makedirs("/dbfs/tmp/movies_project", exist_ok=True)
file_path = "/dbfs/tmp/movies_project/enriched_historical_dramas.json"

with open(file_path, "w") as f:
    json.dump(enriched_movies, f)

print(f"Phase 1 Complete! Saved {len(enriched_movies)} raw nested movies to {file_path}")

## Transformation & Load to Cleaned DBFS (PySpark)

In [0]:
from pyspark.sql.functions import col
from pyspark.sql.types import IntegerType

# read raw data from the previous step
raw_df = spark.read.json("dbfs:/tmp/movies_project/enriched_historical_dramas.json")

# TRANSFORM
transformed_df = raw_df.select(
    col("id").cast(IntegerType()).alias("movie_id"),
    col("title"),
    col("release_date"),
    col("budget").cast(IntegerType()),
    col("revenue").cast(IntegerType()),
    col("vote_average"),
    # Reach into the nested 'genres' array and grab the first dictionary's 'name'
    col("genres")[0]["name"].alias("primary_genre")
)

# clean up rows with release_date=null and budget=0
clean_df = transformed_df.filter(col("release_date").isNotNull() & (col("budget") > 0))

print(f"Transformation complete. Cleaned down to {clean_df.count()} highly curated movies.")

# LOAD TO DELTA TABLE
clean_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("tmdb_historical_dramas_clean")

print("Phase 2 Complete! Data loaded into Delta Table 'tmdb_historical_dramas_clean'.")

In [0]:
# Final Display
display(spark.sql("SELECT * FROM tmdb_historical_dramas_clean ORDER BY revenue DESC LIMIT 10"))